In [1]:

import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('incidence-rate-data.csv')


In [3]:
df1 = pd.read_csv('coverage-data.csv',encoding='latin1')  
df2 = pd.read_csv('vaccine-introduction-data.csv',encoding='latin1')
df3 = pd.read_csv('vaccine-schedule-data.csv',encoding='latin1')
df4 = pd.read_csv('reported-cases-data.csv',encoding='latin1')

In [4]:
df1.isnull().sum()

GROUP                                 0
CODE                                  1
NAME                               1275
YEAR                                  1
ANTIGEN                               1
ANTIGEN_DESCRIPTION                   1
COVERAGE_CATEGORY                     1
COVERAGE_CATEGORY_DESCRIPTION         1
TARGET_NUMBER                    320829
DOSES                            320532
COVERAGE                         169382
dtype: int64

In [5]:
df1.head(20)  

,GROUP,CODE,NAME,YEAR,ANTIGEN,ANTIGEN_DESCRIPTION,COVERAGE_CATEGORY,COVERAGE_CATEGORY_DESCRIPTION,TARGET_NUMBER,DOSES,COVERAGE
0,COUNTRIES,ABW,Aruba,2023.0,BCG,BCG,ADMIN,Administrative coverage,NaN,NaN,NaN
1,COUNTRIES,ABW,Aruba,2023.0,BCG,BCG,OFFICIAL,Official coverage,NaN,NaN,NaN
2,COUNTRIES,ABW,Aruba,2023.0,DIPHCV4,"Diphtheria-containing vaccine, 4th dose (1st b...",ADMIN,Administrative coverage,1044.0,945.0,90.52
3,COUNTRIES,ABW,Aruba,2023.0,DIPHCV4,"Diphtheria-containing vaccine, 4th dose (1st b...",OFFICIAL,Official coverage,NaN,NaN,90.52
4,COUNTRIES,ABW,Aruba,2023.0,DIPHCV5,"Diphtheria-containing vaccine, 5th dose (2nd b...",ADMIN,Administrative coverage,1219.0,1008.0,82.69
5,COUNTRIES,ABW,Aruba,2023.0,DIPHCV5,"Diphtheria-containing vaccine, 5th dose (2nd b...",OFFICIAL,Official coverage,NaN,NaN,82.69
6,COUNTRIES,ABW,Aruba,2023.0,DIPHCV6,"Diphtheria-containing vaccine, 6th dose (3rd b...",ADMIN,Administrative coverage,1468.0,1219.0,83.04
7,COUNTRIES,ABW,Aruba,2023.0,DIPHCV6,"Diphtheria-containing vaccine, 6th dose (3rd b...",OFFICIAL,Official coverage,NaN,NaN,83.04
8,COUNTRIES,ABW,Aruba,2023.0,DTPCV1,"DTP-containing vaccine, 1st dose",ADMIN,Administrative coverage,945.0,926.0,97.99
9,COUNTRIES,ABW,Aruba,2023.0,DTPCV1,"DTP-containing vaccine, 1st dose",OFFICIAL,Official coverage,NaN,NaN,97.99


In [6]:
import pandas as pd
import numpy as np
from collections import Counter

# =============================================================================
# STEP 1: LOAD ALL TABLES
# =============================================================================
# Replace with your actual file paths
coverage_data = pd.read_csv('coverage-data.csv',encoding='latin1')
incidence_rate = pd.read_csv('incidence-rate-data.csv',encoding='latin1')
reported_cases = pd.read_csv('reported-cases-data.csv',encoding='latin1')
vaccine_intro = pd.read_csv('vaccine-introduction-data.csv',encoding='latin1')
vaccine_schedule = pd.read_csv('vaccine-schedule-data.csv',encoding='latin1')

# Store all dataframes in a dictionary for easy iteration
tables = {
    'Coverage Data': coverage_data,
    'Incidence Rate': incidence_rate,
    'Reported Cases': reported_cases,
    'Vaccine Introduction': vaccine_intro,
    'Vaccine Schedule': vaccine_schedule
}

print("="*80)
print("VACCINATION DATA QUALITY ASSESSMENT REPORT")
print("="*80)

# =============================================================================
# STEP 2: DATA TYPES & NULL VALUES ANALYSIS
# =============================================================================
def analyze_data_quality(df, table_name):
    """Analyze data types, null values, and basic statistics"""
    
    print(f"\n{'='*80}")
    print(f"TABLE: {table_name}")
    print(f"{'='*80}")
    
    # Basic info
    print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    
    # Data types and null analysis
    print("\n" + "-"*80)
    print("COLUMN ANALYSIS: Data Types & Null Values")
    print("-"*80)
    print(f"{'Column Name':<30} {'Data Type':<15} {'Null Count':<12} {'Null %':<10} {'Unique'}")
    print("-"*80)
    
    for col in df.columns:
        null_count = df[col].isnull().sum()
        null_pct = (null_count / len(df)) * 100
        unique_count = df[col].nunique()
        
        print(f"{col:<30} {str(df[col].dtype):<15} {null_count:<12,} {null_pct:<10.2f} {unique_count:,}")
    
    # Summary statistics for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        print("\n" + "-"*80)
        print("NUMERIC COLUMNS - SUMMARY STATISTICS")
        print("-"*80)
        print(df[numeric_cols].describe().round(2))
    
    # Check for duplicate rows
    dup_count = df.duplicated().sum()
    print(f"\n{'Total Duplicate Rows:':<30} {dup_count:,}")
    
    return

# =============================================================================
# STEP 3: GRAIN ANALYSIS (What makes each row unique?)
# =============================================================================
def analyze_grain(df, table_name, potential_keys):
    """Identify the grain of the table - what makes each row unique"""
    
    print(f"\n{'='*80}")
    print(f"GRAIN ANALYSIS: {table_name}")
    print(f"{'='*80}")
    
    total_rows = len(df)
    
    print(f"\nTotal Rows: {total_rows:,}")
    print("\nTesting potential key combinations...")
    print("-"*80)
    
    for key_combo in potential_keys:
        # Check if columns exist
        if all(col in df.columns for col in key_combo):
            unique_combos = df[key_combo].drop_duplicates().shape[0]
            is_unique = unique_combos == total_rows
            
            key_str = " + ".join(key_combo)
            status = "✓ UNIQUE (Candidate Primary Key)" if is_unique else "✗ NOT UNIQUE"
            
            print(f"{key_str:<50} {unique_combos:>10,} / {total_rows:,}  {status}")
            
            # If not unique, show some duplicates
            if not is_unique:
                duplicates = df[df.duplicated(subset=key_combo, keep=False)][key_combo]
                if len(duplicates) > 0:
                    print(f"  → Found {len(duplicates):,} duplicate combinations")
    
    return

# =============================================================================
# STEP 4: FOREIGN KEY RELATIONSHIP ANALYSIS
# =============================================================================
def analyze_relationships():
    """Analyze potential foreign key relationships between tables"""
    
    print(f"\n{'='*80}")
    print("FOREIGN KEY RELATIONSHIP ANALYSIS")
    print(f"{'='*80}")
    
    # Define relationship tests
    relationships = [
        ('Coverage Data', 'Code', 'Vaccine Introduction', 'ISO_3_Code', 'Country Code'),
        ('Coverage Data', 'Code', 'Vaccine Schedule', 'ISO_3_Code', 'Country Code'),
        ('Incidence Rate', 'Code', 'Vaccine Introduction', 'ISO_3_Code', 'Country Code'),
        ('Reported Cases', 'Code', 'Vaccine Introduction', 'ISO_3_Code', 'Country Code'),
        ('Coverage Data', 'Code', 'Incidence Rate', 'Code', 'Country Code'),
        ('Coverage Data', 'Code', 'Reported Cases', 'Code', 'Country Code'),
    ]
    
    print("\nTesting referential integrity between tables...")
    print("-"*80)
    
    for source_table, source_col, target_table, target_col, rel_name in relationships:
        if source_table in tables and target_table in tables:
            source_df = tables[source_table]
            target_df = tables[target_table]
            
            if source_col in source_df.columns and target_col in target_df.columns:
                # Get unique values from both
                source_values = set(source_df[source_col].dropna().unique())
                target_values = set(target_df[target_col].dropna().unique())
                
                # Check how many source values exist in target
                matching = source_values.intersection(target_values)
                missing = source_values - target_values
                
                match_pct = (len(matching) / len(source_values) * 100) if len(source_values) > 0 else 0
                
                print(f"\n{rel_name}:")
                print(f"  {source_table}.{source_col} → {target_table}.{target_col}")
                print(f"  Source unique values: {len(source_values):,}")
                print(f"  Target unique values: {len(target_values):,}")
                print(f"  Matching: {len(matching):,} ({match_pct:.1f}%)")
                print(f"  Missing in target: {len(missing):,}")
                
                if len(missing) > 0 and len(missing) <= 10:
                    print(f"  Missing values: {list(missing)[:10]}")

# =============================================================================
# RUN ALL ANALYSES
# =============================================================================

# 1. Data Quality Analysis for each table
for table_name, df in tables.items():
    analyze_data_quality(df, table_name)

# 2. Grain Analysis for each table
print("\n\n" + "="*80)
print("GRAIN ANALYSIS FOR ALL TABLES")
print("="*80)

# Define potential key combinations for each table
grain_tests = {
    'Coverage Data': [
        ['Code', 'Year', 'Antigen'],
        ['Code', 'Year', 'Antigen_description'],
        ['Name', 'Year', 'Antigen'],
        ['Code', 'Year'],
        ['Code', 'Antigen']
    ],
    'Incidence Rate': [
        ['Code', 'Year', 'Disease'],
        ['Name', 'Year', 'Disease'],
        ['Code', 'Year', 'Disease_description'],
        ['Code', 'Year'],
        ['Code', 'Disease']
    ],
    'Reported Cases': [
        ['Code', 'Year', 'Disease'],
        ['Name', 'Year', 'Disease'],
        ['Code', 'Year', 'Disease_description'],
        ['Code', 'Year'],
        ['Code', 'Disease']
    ],
    'Vaccine Introduction': [
        ['ISO_3_Code', 'Year', 'Description'],
        ['Country_Name', 'Year', 'Description'],
        ['ISO_3_Code', 'Description'],
        ['ISO_3_Code', 'Year']
    ],
    'Vaccine Schedule': [
        ['ISO_3_Code', 'Year', 'Vaccine_code', 'Schedule_rounds', 'Target_pop'],
        ['ISO_3_Code', 'Year', 'Vaccine_code', 'Schedule_rounds'],
        ['ISO_3_Code', 'Year', 'Vaccine_description'],
        ['ISO_3_Code', 'Vaccine_code', 'Schedule_rounds']
    ]
}

for table_name, df in tables.items():
    if table_name in grain_tests:
        analyze_grain(df, table_name, grain_tests[table_name])

# 3. Foreign Key Relationship Analysis
analyze_relationships()

# =============================================================================
# STEP 5: DATA QUALITY ISSUES SUMMARY
# =============================================================================
print("\n\n" + "="*80)
print("DATA QUALITY ISSUES SUMMARY")
print("="*80)

for table_name, df in tables.items():
    print(f"\n{table_name}:")
    
    issues = []
    
    # Check for nulls in important columns
    null_cols = df.columns[df.isnull().any()].tolist()
    if null_cols:
        issues.append(f"  • Null values in: {', '.join(null_cols)}")
    
    # Check for duplicates
    if df.duplicated().sum() > 0:
        issues.append(f"  • {df.duplicated().sum():,} duplicate rows found")
    
    # Check for negative values in numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if (df[col] < 0).any():
            issues.append(f"  • Negative values in {col}")
    
    if issues:
        for issue in issues:
            print(issue)
    else:
        print("  ✓ No major issues detected")

print("\n" + "="*80)
print("ASSESSMENT COMPLETE")
print("="*80)
print("\nNext Steps:")
print("1. Review the grain analysis to confirm primary keys")
print("2. Address null values based on your imputation strategy")
print("3. Investigate any referential integrity issues")
print("4. Proceed with data cleaning based on findings")
print("="*80)

VACCINATION DATA QUALITY ASSESSMENT REPORT

TABLE: Coverage Data

Shape: 399,859 rows × 11 columns

--------------------------------------------------------------------------------
COLUMN ANALYSIS: Data Types & Null Values
--------------------------------------------------------------------------------
Column Name                    Data Type       Null Count   Null %     Unique
--------------------------------------------------------------------------------
GROUP                          object          0            0.00       9
CODE                           object          1            0.00       245
NAME                           object          1,275        0.32       242
YEAR                           float64         1            0.00       44
ANTIGEN                        object          1            0.00       69
ANTIGEN_DESCRIPTION            object          1            0.00       69
COVERAGE_CATEGORY              object          1            0.00       5
COVERAGE_CATEGORY_D

In [7]:
import pandas as pd
import numpy as np

# Load data (update paths)
coverage_data = pd.read_csv('coverage-data.csv',encoding='latin1')
incidence_rate = pd.read_csv('incidence-rate-data.csv',encoding='latin1')
reported_cases = pd.read_csv('reported-cases-data.csv',encoding='latin1')
vaccine_intro = pd.read_csv('vaccine-introduction-data.csv',encoding='latin1')
vaccine_schedule = pd.read_csv('vaccine-schedule-data.csv',encoding='latin1')

print("="*80)
print("ENHANCED GRAIN & RELATIONSHIP ANALYSIS")
print("="*80)

# =============================================================================
# PART 1: DETAILED NULL PATTERN ANALYSIS
# =============================================================================
print("\n" + "="*80)
print("PART 1: NULL PATTERN ANALYSIS FOR COVERAGE DATA")
print("="*80)

# Analyze null patterns
coverage_data['has_target'] = coverage_data['TARGET_NUMBER'].notna()
coverage_data['has_doses'] = coverage_data['DOSES'].notna()
coverage_data['has_coverage'] = coverage_data['COVERAGE'].notna()

print("\n1. Completeness by COVERAGE_CATEGORY:")
print("-"*80)
category_completeness = coverage_data.groupby('COVERAGE_CATEGORY').agg({
    'has_target': ['sum', 'mean'],
    'has_doses': ['sum', 'mean'],
    'has_coverage': ['sum', 'mean'],
    'CODE': 'count'
}).round(3)
category_completeness.columns = ['Target_Count', 'Target_%', 'Doses_Count', 'Doses_%', 'Coverage_Count', 'Coverage_%', 'Total_Rows']
category_completeness['Target_%'] *= 100
category_completeness['Doses_%'] *= 100
category_completeness['Coverage_%'] *= 100
print(category_completeness)

print("\n2. Completeness by YEAR (showing every 5 years):")
print("-"*80)
year_completeness = coverage_data.groupby('YEAR').agg({
    'has_target': 'mean',
    'has_doses': 'mean',
    'has_coverage': 'mean',
    'CODE': 'count'
}).round(3)
year_completeness.columns = ['Target_%', 'Doses_%', 'Coverage_%', 'Total_Rows']
year_completeness *= [100, 100, 100, 1]
# Show every 5 years
years_to_show = [1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2023]
print(year_completeness[year_completeness.index.isin(years_to_show)])

print("\n3. Missing Data Patterns:")
print("-"*80)
coverage_data['pattern'] = (
    coverage_data['has_target'].astype(str) + "_" +
    coverage_data['has_doses'].astype(str) + "_" +
    coverage_data['has_coverage'].astype(str)
)
pattern_map = {
    'True_True_True': 'All Complete',
    'False_False_True': 'Only Coverage%',
    'False_False_False': 'All Missing',
    'True_True_False': 'Target+Doses Only',
}
pattern_counts = coverage_data['pattern'].value_counts()
print("\nPattern Distribution:")
for pattern, count in pattern_counts.head(10).items():
    readable = pattern_map.get(pattern, pattern)
    pct = (count / len(coverage_data)) * 100
    print(f"  {readable:<25} {count:>10,} ({pct:>5.1f}%)")

# =============================================================================
# PART 2: INVALID VALUES INVESTIGATION
# =============================================================================
print("\n" + "="*80)
print("PART 2: INVALID VALUES ANALYSIS")
print("="*80)

print("\n1. Negative DOSES:")
print("-"*80)
negative_doses = coverage_data[coverage_data['DOSES'] < 0]
if len(negative_doses) > 0:
    print(f"Found {len(negative_doses):,} rows with negative DOSES")
    print("\nSample of negative values:")
    print(negative_doses[['CODE', 'NAME', 'YEAR', 'ANTIGEN', 'DOSES']].head(10))
    print(f"\nRange: {negative_doses['DOSES'].min():,.0f} to {negative_doses['DOSES'].max():,.0f}")
else:
    print("✓ No negative DOSES found")

print("\n2. Coverage > 100%:")
print("-"*80)
high_coverage = coverage_data[coverage_data['COVERAGE'] > 100]
if len(high_coverage) > 0:
    print(f"Found {len(high_coverage):,} rows with COVERAGE > 100%")
    
    # Breakdown by severity
    severe = high_coverage[high_coverage['COVERAGE'] > 150]
    moderate = high_coverage[(high_coverage['COVERAGE'] > 100) & (high_coverage['COVERAGE'] <= 150)]
    
    print(f"  • 100-150% (moderate): {len(moderate):,} rows")
    print(f"  • >150% (severe errors): {len(severe):,} rows")
    
    print("\nTop 10 extreme values:")
    print(high_coverage.nlargest(10, 'COVERAGE')[['CODE', 'NAME', 'YEAR', 'ANTIGEN', 'COVERAGE']])
else:
    print("✓ All coverage values ≤ 100%")

print("\n3. Zero or Null TARGET_NUMBER with non-zero DOSES:")
print("-"*80)
weird_doses = coverage_data[
    ((coverage_data['TARGET_NUMBER'] == 0) | (coverage_data['TARGET_NUMBER'].isna())) &
    (coverage_data['DOSES'] > 0)
]
print(f"Found {len(weird_doses):,} rows with doses but no target")
if len(weird_doses) > 0:
    print("Sample:")
    print(weird_doses[['CODE', 'NAME', 'YEAR', 'ANTIGEN', 'TARGET_NUMBER', 'DOSES', 'COVERAGE']].head(5))

# =============================================================================
# PART 3: PRIMARY KEY VERIFICATION (GRAIN ANALYSIS)
# =============================================================================
print("\n" + "="*80)
print("PART 3: PRIMARY KEY VERIFICATION (GRAIN ANALYSIS)")
print("="*80)

def verify_primary_key(df, key_cols, table_name):
    """Verify if column combination is a unique primary key"""
    print(f"\n{table_name}:")
    print("-"*80)
    
    total_rows = len(df)
    
    # Test the primary key
    unique_count = df[key_cols].drop_duplicates().shape[0]
    is_unique = unique_count == total_rows
    
    key_str = " + ".join(key_cols)
    print(f"Testing: {key_str}")
    print(f"  Total rows: {total_rows:,}")
    print(f"  Unique combinations: {unique_count:,}")
    print(f"  Status: {'✓ UNIQUE - Valid Primary Key' if is_unique else '✗ NOT UNIQUE - Contains Duplicates'}")
    
    if not is_unique:
        # Show duplicates
        duplicates = df[df.duplicated(subset=key_cols, keep=False)].sort_values(key_cols)
        print(f"\n  Found {len(duplicates):,} rows with duplicate keys")
        print(f"  Number of duplicate key combinations: {len(duplicates) - unique_count:,}")
        
        # Show example
        if len(duplicates) > 0:
            print("\n  Example duplicates (first 5):")
            print(duplicates[key_cols + [df.columns[-1]]].head(10))
    
    return is_unique

# Test Coverage Data
verify_primary_key(
    coverage_data,
    ['CODE', 'YEAR', 'ANTIGEN'],
    'Coverage Data'
)

# Test Incidence Rate
verify_primary_key(
    incidence_rate,
    ['CODE', 'YEAR', 'DISEASE'],
    'Incidence Rate'
)

# Test Reported Cases
verify_primary_key(
    reported_cases,
    ['CODE', 'YEAR', 'DISEASE'],
    'Reported Cases'
)

# Test Vaccine Introduction
verify_primary_key(
    vaccine_intro,
    ['ISO_3_CODE', 'YEAR', 'DESCRIPTION'],
    'Vaccine Introduction'
)

# Test Vaccine Schedule (might need additional columns)
print("\nVaccine Schedule - Testing multiple key combinations:")
print("-"*80)
schedule_keys = [
    ['ISO_3_CODE', 'YEAR', 'VACCINECODE', 'SCHEDULEROUNDS'],
    ['ISO_3_CODE', 'YEAR', 'VACCINECODE', 'SCHEDULEROUNDS', 'TARGETPOP'],
    ['ISO_3_CODE', 'VACCINECODE', 'SCHEDULEROUNDS', 'TARGETPOP'],
]

for keys in schedule_keys:
    total = len(vaccine_schedule)
    unique = vaccine_schedule[keys].drop_duplicates().shape[0]
    is_unique = unique == total
    key_str = " + ".join(keys)
    status = "✓ UNIQUE" if is_unique else "✗ NOT UNIQUE"
    print(f"{key_str:<70} {unique:>7,} / {total:>7,}  {status}")

# =============================================================================
# PART 4: FOREIGN KEY RELATIONSHIP ANALYSIS
# =============================================================================
print("\n" + "="*80)
print("PART 4: FOREIGN KEY RELATIONSHIP ANALYSIS")
print("="*80)

def analyze_fk_relationship(source_df, source_col, target_df, target_col, description):
    """Analyze foreign key relationship between tables"""
    
    source_values = set(source_df[source_col].dropna().unique())
    target_values = set(target_df[target_col].dropna().unique())
    
    matching = source_values.intersection(target_values)
    missing = source_values - target_values
    
    match_pct = (len(matching) / len(source_values) * 100) if len(source_values) > 0 else 0
    
    print(f"\n{description}:")
    print(f"  Source unique values: {len(source_values):,}")
    print(f"  Target unique values: {len(target_values):,}")
    print(f"  Matching: {len(matching):,} ({match_pct:.1f}%)")
    print(f"  Missing in target: {len(missing):,}")
    
    if 0 < len(missing) <= 20:
        print(f"  Missing codes: {sorted(list(missing))}")
    
    # Recommendation
    if match_pct >= 98:
        print(f"  ✓ Recommendation: Safe to create FOREIGN KEY constraint")
    elif match_pct >= 90:
        print(f"  ⚠ Recommendation: Review missing values, may use without constraint")
    else:
        print(f"  ✗ Recommendation: Data quality issue - investigate before creating relationship")
    
    return match_pct

print("\n1. Country Code Relationships:")
print("-"*80)

# Coverage to Vaccine Introduction
analyze_fk_relationship(
    coverage_data, 'CODE',
    vaccine_intro, 'ISO_3_CODE',
    'Coverage → Vaccine Intro (Country Codes)'
)

# Coverage to Incidence Rate
analyze_fk_relationship(
    coverage_data, 'CODE',
    incidence_rate, 'CODE',
    'Coverage → Incidence Rate (Country Codes)'
)

# Coverage to Reported Cases
analyze_fk_relationship(
    coverage_data, 'CODE',
    reported_cases, 'CODE',
    'Coverage → Reported Cases (Country Codes)'
)

print("\n2. Cross-table Year Consistency:")
print("-"*80)

tables_years = {
    'Coverage': set(coverage_data['YEAR'].dropna().unique()),
    'Incidence': set(incidence_rate['YEAR'].dropna().unique()),
    'Cases': set(reported_cases['YEAR'].dropna().unique()),
    'Intro': set(vaccine_intro['YEAR'].dropna().unique()),
    'Schedule': set(vaccine_schedule['YEAR'].dropna().unique()),
}

for name, years in tables_years.items():
    print(f"  {name:<15} Years: {int(min(years))} to {int(max(years))} (total: {len(years)})")

common_years = set.intersection(*tables_years.values())
print(f"\n  Common years across all tables: {len(common_years)} years")
print(f"  Range: {int(min(common_years))} to {int(max(common_years))}")

# =============================================================================
# PART 5: SUMMARY RECOMMENDATIONS
# =============================================================================
print("\n" + "="*80)
print("SUMMARY & RECOMMENDATIONS")
print("="*80)

print("\n1. PRIMARY KEYS TO USE IN SQL:")
print("-"*80)
print("""
CREATE TABLE coverage_data (
    code VARCHAR(3),
    year INT,
    antigen VARCHAR(50),
    ...
    PRIMARY KEY (code, year, antigen)
);

CREATE TABLE incidence_rate (
    code VARCHAR(3),
    year INT,
    disease VARCHAR(50),
    ...
    PRIMARY KEY (code, year, disease)
);

CREATE TABLE reported_cases (
    code VARCHAR(3),
    year INT,
    disease VARCHAR(50),
    ...
    PRIMARY KEY (code, year, disease)
);

CREATE TABLE vaccine_introduction (
    iso_3_code VARCHAR(3),
    year INT,
    description VARCHAR(100),
    ...
    PRIMARY KEY (iso_3_code, year, description)
);

CREATE TABLE vaccine_schedule (
    iso_3_code VARCHAR(3),
    year INT,
    vaccine_code VARCHAR(50),
    schedule_rounds INT,
    target_pop VARCHAR(50),
    ...
    PRIMARY KEY (iso_3_code, year, vaccine_code, schedule_rounds, target_pop)
);
""")

print("\n2. DATA CLEANING PRIORITY:")
print("-"*80)
print("""
HIGH PRIORITY:
  ✓ Fix negative DOSES values
  ✓ Cap COVERAGE at 100% (or handle >150% as errors)
  ✓ Drop 1-5 rows with null in primary key columns
  
MEDIUM PRIORITY:
  • Analyze 80% missing TARGET/DOSES pattern by COVERAGE_CATEGORY
  • Decide: Keep rows with COVERAGE% only, or drop incomplete rows
  • Standardize country codes (uppercase, trim)
  
LOW PRIORITY:
  • Handle 23-27% nulls in INCIDENCE_RATE and CASES (likely "no data")
  • Review 52% nulls in TARGETPOP (Vaccine Schedule)
""")

print("\n3. FOREIGN KEY RECOMMENDATIONS:")
print("-"*80)
print("""
Based on country code matching analysis above:
  • If match rate > 98%: Create FOREIGN KEY constraint
  • If match rate 90-98%: Create without constraint, document exceptions
  • If match rate < 90%: Investigate data quality issue first
  
Suggested normalized structure:
  • Create dim_countries table (code, name, who_region)
  • Create dim_years table (year)
  • Create dim_vaccines table (antigen, description)
  • Create dim_diseases table (disease, description)
  • Link fact tables to dimensions via foreign keys
""")

print("\n" + "="*80)
print("ANALYSIS COMPLETE - Review results above")
print("="*80)

ENHANCED GRAIN & RELATIONSHIP ANALYSIS

PART 1: NULL PATTERN ANALYSIS FOR COVERAGE DATA

1. Completeness by COVERAGE_CATEGORY:
--------------------------------------------------------------------------------
                   Target_Count  Target_%  Doses_Count  Doses_%  \
COVERAGE_CATEGORY                                                 
ADMIN                     57148      36.7        57496     37.0   
HPV                        6536      25.8         6536     25.8   
OFFICIAL                      1       0.0            0      0.0   
PAB                         704      13.1          654     12.2   
WUENIC                    14641      15.3        14641     15.3   

                   Coverage_Count  Coverage_%  Total_Rows  
COVERAGE_CATEGORY                                          
ADMIN                       72030        46.3      155576  
HPV                          7630        30.2       25304  
OFFICIAL                    69881        59.2      118050  
PAB                   

In [8]:
import pandas as pd
import numpy as np
from datetime import datetime

print("="*80)
print("VACCINATION DATA CLEANING SCRIPT")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================
print("\nSTEP 1: Loading data files...")

coverage_data = pd.read_csv('coverage-data.csv',encoding='latin1')
incidence_rate = pd.read_csv('incidence-rate-data.csv',encoding='latin1')
reported_cases = pd.read_csv('reported-cases-data.csv',encoding='latin1')
vaccine_intro = pd.read_csv('vaccine-introduction-data.csv',encoding='latin1')
vaccine_schedule = pd.read_csv('vaccine-schedule-data.csv',encoding='latin1')


print(f"✓ Coverage Data: {len(coverage_data):,} rows loaded")
print(f"✓ Incidence Rate: {len(incidence_rate):,} rows loaded")
print(f"✓ Reported Cases: {len(reported_cases):,} rows loaded")
print(f"✓ Vaccine Introduction: {len(vaccine_intro):,} rows loaded")
print(f"✓ Vaccine Schedule: {len(vaccine_schedule):,} rows loaded")

# Store original counts for reporting
original_counts = {
    'coverage': len(coverage_data),
    'incidence': len(incidence_rate),
    'cases': len(reported_cases),
    'intro': len(vaccine_intro),
    'schedule': len(vaccine_schedule)
}

# =============================================================================
# STEP 2: CLEAN COVERAGE DATA
# =============================================================================
print("\n" + "="*80)
print("STEP 2: CLEANING COVERAGE DATA")
print("="*80)

# 2.1: Drop rows with null in primary key columns
print("\n2.1: Removing rows with NULL in key columns...")
before = len(coverage_data)
coverage_data = coverage_data.dropna(subset=['CODE', 'YEAR', 'ANTIGEN', 'COVERAGE_CATEGORY'])
after = len(coverage_data)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

# 2.2: Fix negative DOSES
print("\n2.2: Fixing negative DOSES values...")
negative_mask = coverage_data['DOSES'] < 0
negative_count = negative_mask.sum()
if negative_count > 0:
    print(f"  Found {negative_count} negative values")
    print(f"  Range: {coverage_data.loc[negative_mask, 'DOSES'].min():,.0f} to {coverage_data.loc[negative_mask, 'DOSES'].max():,.0f}")
    # Set negative values to NULL
    coverage_data.loc[negative_mask, 'DOSES'] = np.nan
    print(f"  ✓ Converted to NULL")
else:
    print(f"  ✓ No negative values found")

# 2.3: Fix impossible COVERAGE percentages
print("\n2.3: Fixing impossible COVERAGE percentages...")

# Count issues
severe_mask = coverage_data['COVERAGE'] > 150
moderate_mask = (coverage_data['COVERAGE'] > 100) & (coverage_data['COVERAGE'] <= 150)

severe_count = severe_mask.sum()
moderate_count = moderate_mask.sum()

print(f"  Severe errors (>150%): {severe_count:,} rows")
print(f"  Moderate (100-150%): {moderate_count:,} rows")

# Fix severe errors (>150%) - set to NULL
if severe_count > 0:
    coverage_data.loc[severe_mask, 'COVERAGE'] = np.nan
    print(f"  ✓ Set >150% to NULL")

# Fix moderate overcounting (100-150%) - cap at 100%
if moderate_count > 0:
    coverage_data.loc[moderate_mask, 'COVERAGE'] = 100.0
    print(f"  ✓ Capped 100-150% values at 100%")

# 2.4: Standardize text columns
print("\n2.4: Standardizing text columns...")
coverage_data['CODE'] = coverage_data['CODE'].str.upper().str.strip()
coverage_data['ANTIGEN'] = coverage_data['ANTIGEN'].str.upper().str.strip()
coverage_data['COVERAGE_CATEGORY'] = coverage_data['COVERAGE_CATEGORY'].str.upper().str.strip()
print(f"  ✓ Standardized CODE, ANTIGEN, COVERAGE_CATEGORY")

# 2.5: Convert YEAR to integer (it's currently float)
print("\n2.5: Converting YEAR to integer...")
coverage_data['YEAR'] = coverage_data['YEAR'].astype(int)
print(f"  ✓ Converted YEAR from float to int")

# 2.6: Add data quality flags
print("\n2.6: Adding data quality flags...")
coverage_data['has_admin_data'] = (
    coverage_data['TARGET_NUMBER'].notna() & 
    coverage_data['DOSES'].notna()
)
coverage_data['data_completeness'] = np.where(
    coverage_data['has_admin_data'] & coverage_data['COVERAGE'].notna(), 'Complete',
    np.where(coverage_data['COVERAGE'].notna(), 'Coverage_Only',
    np.where(coverage_data['has_admin_data'], 'Admin_Only', 'Missing'))
)
print(f"  ✓ Added has_admin_data and data_completeness flags")

# =============================================================================
# STEP 3: CLEAN INCIDENCE RATE DATA
# =============================================================================
print("\n" + "="*80)
print("STEP 3: CLEANING INCIDENCE RATE DATA")
print("="*80)

print("\n3.1: Removing rows with NULL in key columns...")
before = len(incidence_rate)
incidence_rate = incidence_rate.dropna(subset=['CODE', 'YEAR', 'DISEASE'])
after = len(incidence_rate)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

print("\n3.2: Standardizing text columns...")
incidence_rate['CODE'] = incidence_rate['CODE'].str.upper().str.strip()
incidence_rate['DISEASE'] = incidence_rate['DISEASE'].str.upper().str.strip()
print(f"  ✓ Standardized CODE, DISEASE")

print("\n3.3: Converting YEAR to integer...")
incidence_rate['YEAR'] = incidence_rate['YEAR'].astype(int)
print(f"  ✓ Converted YEAR")

print("\n3.4: Checking for negative incidence rates...")
negative_count = (incidence_rate['INCIDENCE_RATE'] < 0).sum()
if negative_count > 0:
    incidence_rate.loc[incidence_rate['INCIDENCE_RATE'] < 0, 'INCIDENCE_RATE'] = np.nan
    print(f"  Fixed {negative_count} negative values")
else:
    print(f"  ✓ No negative values found")

# =============================================================================
# STEP 4: CLEAN REPORTED CASES DATA
# =============================================================================
print("\n" + "="*80)
print("STEP 4: CLEANING REPORTED CASES DATA")
print("="*80)

print("\n4.1: Removing rows with NULL in key columns...")
before = len(reported_cases)
reported_cases = reported_cases.dropna(subset=['CODE', 'YEAR', 'DISEASE'])
after = len(reported_cases)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

print("\n4.2: Standardizing text columns...")
reported_cases['CODE'] = reported_cases['CODE'].str.upper().str.strip()
reported_cases['DISEASE'] = reported_cases['DISEASE'].str.upper().str.strip()
print(f"  ✓ Standardized CODE, DISEASE")

print("\n4.3: Converting YEAR to integer...")
reported_cases['YEAR'] = reported_cases['YEAR'].astype(int)
print(f"  ✓ Converted YEAR")

print("\n4.4: Checking for negative case counts...")
negative_count = (reported_cases['CASES'] < 0).sum()
if negative_count > 0:
    reported_cases.loc[reported_cases['CASES'] < 0, 'CASES'] = np.nan
    print(f"  Fixed {negative_count} negative values")
else:
    print(f"  ✓ No negative values found")

# =============================================================================
# STEP 5: CLEAN VACCINE INTRODUCTION DATA
# =============================================================================
print("\n" + "="*80)
print("STEP 5: CLEANING VACCINE INTRODUCTION DATA")
print("="*80)

print("\n5.1: Removing rows with NULL in key columns...")
before = len(vaccine_intro)
vaccine_intro = vaccine_intro.dropna(subset=['ISO_3_CODE', 'YEAR', 'DESCRIPTION'])
after = len(vaccine_intro)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

print("\n5.2: Standardizing text columns...")
vaccine_intro['ISO_3_CODE'] = vaccine_intro['ISO_3_CODE'].str.upper().str.strip()
vaccine_intro['DESCRIPTION'] = vaccine_intro['DESCRIPTION'].str.upper().str.strip()
print(f"  ✓ Standardized ISO_3_CODE, DESCRIPTION")

print("\n5.3: Converting YEAR to integer...")
vaccine_intro['YEAR'] = vaccine_intro['YEAR'].astype(int)
print(f"  ✓ Converted YEAR")

# =============================================================================
# STEP 6: CLEAN VACCINE SCHEDULE DATA
# =============================================================================
print("\n" + "="*80)
print("STEP 6: CLEANING VACCINE SCHEDULE DATA")
print("="*80)

print("\n6.1: Removing rows with NULL in key columns...")
before = len(vaccine_schedule)
vaccine_schedule = vaccine_schedule.dropna(subset=['ISO_3_CODE', 'YEAR', 'VACCINECODE', 'SCHEDULEROUNDS'])
after = len(vaccine_schedule)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

print("\n6.2: Standardizing text columns...")
vaccine_schedule['ISO_3_CODE'] = vaccine_schedule['ISO_3_CODE'].str.upper().str.strip()
vaccine_schedule['VACCINECODE'] = vaccine_schedule['VACCINECODE'].str.upper().str.strip()
print(f"  ✓ Standardized ISO_3_CODE, VACCINECODE")

print("\n6.3: Converting YEAR to integer...")
vaccine_schedule['YEAR'] = vaccine_schedule['YEAR'].astype(int)
print(f"  ✓ Converted YEAR")

# =============================================================================
# STEP 7: VERIFY PRIMARY KEYS AFTER CLEANING
# =============================================================================
print("\n" + "="*80)
print("STEP 7: VERIFYING PRIMARY KEYS")
print("="*80)

def verify_pk(df, cols, name):
    total = len(df)
    unique = df[cols].drop_duplicates().shape[0]
    is_unique = total == unique
    status = "✓ UNIQUE" if is_unique else "✗ NOT UNIQUE"
    print(f"{name:<30} {unique:>10,} / {total:>10,}  {status}")
    return is_unique

print("\nPrimary Key Verification:")
print("-"*80)
pk_coverage = verify_pk(coverage_data, ['CODE', 'YEAR', 'ANTIGEN', 'COVERAGE_CATEGORY'], 'Coverage Data')
pk_incidence = verify_pk(incidence_rate, ['CODE', 'YEAR', 'DISEASE'], 'Incidence Rate')
pk_cases = verify_pk(reported_cases, ['CODE', 'YEAR', 'DISEASE'], 'Reported Cases')
pk_intro = verify_pk(vaccine_intro, ['ISO_3_CODE', 'YEAR', 'DESCRIPTION'], 'Vaccine Introduction')
pk_schedule = verify_pk(vaccine_schedule, ['ISO_3_CODE', 'YEAR', 'VACCINECODE', 'SCHEDULEROUNDS', 'TARGETPOP'], 'Vaccine Schedule')

# =============================================================================
# STEP 8: GENERATE CLEANING SUMMARY REPORT
# =============================================================================
print("\n" + "="*80)
print("STEP 8: CLEANING SUMMARY REPORT")
print("="*80)

summary_data = {
    'Table': ['Coverage Data', 'Incidence Rate', 'Reported Cases', 'Vaccine Introduction', 'Vaccine Schedule'],
    'Original_Rows': [original_counts['coverage'], original_counts['incidence'], original_counts['cases'], original_counts['intro'], original_counts['schedule']],
    'Cleaned_Rows': [len(coverage_data), len(incidence_rate), len(reported_cases), len(vaccine_intro), len(vaccine_schedule)],
    'Rows_Removed': [
        original_counts['coverage'] - len(coverage_data),
        original_counts['incidence'] - len(incidence_rate),
        original_counts['cases'] - len(reported_cases),
        original_counts['intro'] - len(vaccine_intro),
        original_counts['schedule'] - len(vaccine_schedule)
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df['Retention_%'] = (summary_df['Cleaned_Rows'] / summary_df['Original_Rows'] * 100).round(2)

print("\n" + summary_df.to_string(index=False))

print("\n\nData Quality Improvements:")
print("-"*80)
print(f"✓ Fixed {negative_mask.sum():,} negative DOSES values")
print(f"✓ Fixed {severe_count:,} severe coverage errors (>150%)")
print(f"✓ Capped {moderate_count:,} moderate overcounting (100-150%)")
print(f"✓ Standardized all country codes and text fields")
print(f"✓ Converted YEAR columns from float to integer")
print(f"✓ Added data quality flags to coverage data")

# =============================================================================
# STEP 9: EXPORT CLEANED DATA
# =============================================================================
print("\n" + "="*80)
print("STEP 9: EXPORTING CLEANED DATA")
print("="*80)

output_folder = 'cleaned_data/'
import os
os.makedirs(output_folder, exist_ok=True)

# Export to CSV
coverage_data.to_csv(f'{output_folder}coverage_data_cleaned.csv', index=False)
incidence_rate.to_csv(f'{output_folder}incidence_rate_cleaned.csv', index=False)
reported_cases.to_csv(f'{output_folder}reported_cases_cleaned.csv', index=False)
vaccine_intro.to_csv(f'{output_folder}vaccine_introduction_cleaned.csv', index=False)
vaccine_schedule.to_csv(f'{output_folder}vaccine_schedule_cleaned.csv', index=False)

print(f"\n✓ All cleaned files exported to '{output_folder}' folder:")
print(f"  • coverage_data_cleaned.csv")
print(f"  • incidence_rate_cleaned.csv")
print(f"  • reported_cases_cleaned.csv")
print(f"  • vaccine_introduction_cleaned.csv")
print(f"  • vaccine_schedule_cleaned.csv")

# Export summary report
summary_df.to_csv(f'{output_folder}cleaning_summary_report.csv', index=False)
print(f"  • cleaning_summary_report.csv")

print("\n" + "="*80)
print("CLEANING COMPLETE!")
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

print("\n" + "="*80)
print("NEXT STEPS")
print("="*80)


VACCINATION DATA CLEANING SCRIPT
Started: 2026-01-13 16:58:21

STEP 1: Loading data files...
✓ Coverage Data: 399,859 rows loaded
✓ Incidence Rate: 84,946 rows loaded
✓ Reported Cases: 84,870 rows loaded
✓ Vaccine Introduction: 138,321 rows loaded
✓ Vaccine Schedule: 8,053 rows loaded

STEP 2: CLEANING COVERAGE DATA

2.1: Removing rows with NULL in key columns...
  Removed: 1 rows
  Remaining: 399,858 rows

2.2: Fixing negative DOSES values...
  Found 8 negative values
  Range: -222,288,203 to -3,333
  ✓ Converted to NULL

2.3: Fixing impossible COVERAGE percentages...
  Severe errors (>150%): 95 rows
  Moderate (100-150%): 5,002 rows
  ✓ Set >150% to NULL
  ✓ Capped 100-150% values at 100%

2.4: Standardizing text columns...
  ✓ Standardized CODE, ANTIGEN, COVERAGE_CATEGORY

2.5: Converting YEAR to integer...
  ✓ Converted YEAR from float to int

2.6: Adding data quality flags...
  ✓ Added has_admin_data and data_completeness flags

STEP 3: CLEANING INCIDENCE RATE DATA

3.1: Removing 